In [ ]:
# Perform Sementic Search model

# Install langchain




In [1]:

# laod the data
data= """The Magic Library is located in the clouds. To enter the library, you must whisper the secret password 'Lumos'. Once inside, you can find books that talk.
 Some books are friendly, while others are very grumpy. Do not wake the grumpy books up."""

In [4]:
!pip install -qU langchain-text-splitters

In [5]:
# split the text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=20,
    length_function=len,
)
chunks=splitter.split_text(data)
chunks

["The Magic Library is located in the clouds. To enter the library, you must whisper the secret password 'Lumos'. Once inside, you can find books that talk.",
 'Some books are friendly, while others are very grumpy. Do not wake the grumpy books up.']

In [9]:
# Transform the chunks of data into embdeddings
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")

# This model has 256 tokens per context window


embeddings=model.encode(chunks,batch_size=32,show_progress_bar=True)
embeddings


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([[ 6.46093255e-03, -9.41820145e-02, -7.97962248e-02,
        -1.00302054e-02,  1.18019572e-02, -2.11131517e-02,
        -1.15224048e-02, -3.81907076e-02,  4.07355018e-02,
        -2.79080532e-02, -2.71089729e-02,  9.25084725e-02,
        -4.55761999e-02, -9.88080353e-02,  2.35502850e-02,
        -1.07063269e-02, -2.73768120e-02,  3.52173001e-02,
        -7.63612241e-03, -4.40554433e-02,  1.07272595e-01,
        -9.21292324e-03,  3.51533405e-02,  2.45508105e-02,
        -5.09576090e-02, -1.53453900e-02, -3.29716057e-02,
        -4.70165946e-02, -5.33762574e-02, -1.71942525e-02,
         2.15914156e-02,  2.81526428e-02, -2.67833453e-02,
        -3.96593399e-02, -8.93366151e-03,  7.36980438e-02,
         4.57919389e-02, -3.34514566e-02,  2.47570816e-02,
        -8.03424418e-02, -5.46618141e-02,  2.86232270e-02,
        -6.45018443e-02,  4.24396545e-02, -7.08252117e-02,
        -7.75849121e-03,  1.94751061e-02, -5.54223023e-02,
         2.01037508e-02, -9.06955451e-03, -4.74815071e-0

In [10]:
#Install weviate
!pip install -U weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.7 MB/s eta 0:00:00


In [15]:
# store the embdedding in vector db
from google.colab import userdata
import weaviate
import os
import weaviate.classes.config as wvc
from weaviate.auth import AuthApiKey
# from weaviate.util import generate_uuid5
weaviate_url = userdata.get("WEAVIATE_URL")
print(weaviate_url)

weaviate_api_key = userdata.get("WEAVIATE_API_KEY")
print(weaviate_api_key)
if len(chunks) != len(embeddings):
  raise Exception("The number of chunks and embeddings must be the same")
# connect to wevaite
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=AuthApiKey(weaviate_api_key),
) as client:

  dataResult=client.collections.create(
      name="MagicLibrary",
      vector_config=None,
      properties=[
        wvc.Property(name="content", data_type=wvc.DataType.TEXT),
        wvc.Property(name="source", data_type=wvc.DataType.TEXT),
    ]
  )
  # now store the data in weviate
  with dataResult.batch.dynamic() as batch:
    for i,chunk in enumerate(chunks):
      # create the propretires
      properties={
          "content":chunk,
          "source":f"source_{i}"
      }
      # send the data the vector dv
      batch.add_object(
          properties,
         vector=embeddings[i]
      )



In [16]:
# make semlarity search
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:
  library=client.collections.get("MagicLibrary")
  userInput="where is the magic library located?"
  # embend user input
  userInputEmbended=model.encode(userInput).tolist()
  # make the query
  response=library.query.near_vector(
      near_vector=userInputEmbended,
      limit=2
  )
  for obj in response.objects:
    print(f"Match found: {obj.properties['content']}")

Match found: The Magic Library is located in the clouds. To enter the library, you must whisper the secret password 'Lumos'. Once inside, you can find books that talk.
Match found: Some books are friendly, while others are very grumpy. Do not wake the grumpy books up.
